# Baseline Zero-/Few-Shot Eval: Qwen2.5-VL on Chessboard Understanding

Evaluates a vanilla (non-fine-tuned) Qwen2.5-VL against the 3 task datasets pushed to the Hugging Face Hub (`bdatm-project/chess-task1/2/3`), streamed directly rather than downloaded in full. This establishes the baseline numbers that later LoRA fine-tuning and patch-reordering experiments get compared against.

Set `N_SHOT = 0` for zero-shot, or `N_SHOT > 0` for few-shot (examples are drawn from `train`, evaluation always runs on `test`).

In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils datasets huggingface_hub python-chess


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import torch
from tqdm import tqdm


In [ ]:
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}


## Clone the repo (for `src/eval/metrics.py` and `src/data/prompts`)

In [ ]:
if CONFIG["colab"]:
    repo_dir = Path(CONFIG["repo_dir"])
    if repo_dir.exists():
        subprocess.run(["rm", "-rf", str(repo_dir)], check=True)

    repo_url = f"https://github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_dir)],
        capture_output=True, text=True
    )
    assert result.returncode == 0, f"Git clone failed: {result.stderr}"

    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path(".").resolve()
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))

REPO_ROOT = Path(".").resolve()
print("Setup complete. REPO_ROOT:", REPO_ROOT)


In [ ]:
from src.eval.metrics import (
    fen_exact_match,
    character_error_rate,
    square_by_square_accuracy,
    san_exact_match,
)
from src.data.utilities import authenticate_hf


## Authenticate with Hugging Face

Add your token once as a Colab secret (key icon in the left sidebar) named `HF_TOKEN` and toggle "Notebook access" on. Read access is enough here (no push in this notebook).

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN found; continuing unauthenticated (fine if the datasets are public).")


## Load Qwen2.5-VL

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print("Model loaded:", MODEL_ID)


## Config: task, split sizes, shot count

In [ ]:
ORGANIZATION_NAME = "bdatm-project"
TASKS = ["task1", "task2", "task3"]
N_EVAL = 100       # number of test examples to score per task (keep small first; streaming makes this cheap to raise later)
N_SHOT = 0         # 0 = zero-shot; >0 = few-shot, examples drawn from `train`
MAX_NEW_TOKENS = 96


## Inference helpers

In [ ]:
from qwen_vl_utils import process_vision_info
from datasets import load_dataset


def sample_to_content(sample, task):
    """Turns one streamed HF sample into the image content blocks for a chat message."""
    if task == "task3":
        return [
            {"type": "image", "image": sample["image_t"]},
            {"type": "image", "image": sample["image_t1"]},
        ]
    return [{"type": "image", "image": sample["image"]}]


def build_messages(task, prompt, test_sample, few_shot_samples):
    """
    Builds a single-turn chat with N few-shot (image(s) -> target) exchanges
    prepended before the actual test sample, all sharing the same task prompt
    as system instruction.
    """
    messages = [{"role": "system", "content": prompt}]

    for shot in few_shot_samples:
        messages.append({"role": "user", "content": sample_to_content(shot, task)})
        messages.append({"role": "assistant", "content": shot["target"]})

    messages.append({"role": "user", "content": sample_to_content(test_sample, task)})
    return messages


@torch.no_grad()
def generate(messages):
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]
    return output.strip()


## Run zero-/few-shot eval per task

In [ ]:
def score_task1(predictions, targets):
    fen_em = sum(fen_exact_match(p, t) for p, t in zip(predictions, targets)) / len(targets)
    cer = sum(character_error_rate(p, t) for p, t in zip(predictions, targets)) / len(targets)
    sq_acc = sum(square_by_square_accuracy(p, t) for p, t in zip(predictions, targets)) / len(targets)
    return {"fen_exact_match": fen_em, "character_error_rate": cer, "square_by_square_accuracy": sq_acc}


def score_san_task(predictions, targets):
    em = sum(san_exact_match(p, t) for p, t in zip(predictions, targets)) / len(targets)
    return {"exact_match": em}


results = {}

for task in TASKS:
    repo_id = f"{ORGANIZATION_NAME}/chess-{task}"
    test_stream = load_dataset(repo_id, split="test", streaming=True)

    few_shot_samples = []
    if N_SHOT > 0:
        train_stream = load_dataset(repo_id, split="train", streaming=True)
        few_shot_samples = list(train_stream.take(N_SHOT))

    prompt = None
    predictions, targets = [], []

    for sample in tqdm(test_stream.take(N_EVAL), total=N_EVAL, desc=f"{task} ({'zero' if N_SHOT == 0 else N_SHOT}-shot)"):
        prompt = sample["prompt"]
        messages = build_messages(task, prompt, sample, few_shot_samples)
        prediction = generate(messages)
        predictions.append(prediction)
        targets.append(sample["target"])

    if task == "task1":
        metrics = score_task1(predictions, targets)
    else:
        metrics = score_san_task(predictions, targets)

    results[task] = {
        "n_shot": N_SHOT,
        "n_eval": len(targets),
        "metrics": metrics,
        "predictions": predictions,
        "targets": targets,
    }
    print(task, metrics)


## Save results (for later comparison against fine-tuned / reordered runs)

In [ ]:
from src.data.serialization import save_json

out_path = Path(f"baseline_eval_{MODEL_ID.split('/')[-1]}_{N_SHOT}shot.json")
save_json({"model_id": MODEL_ID, **results}, out_path)
print("Saved to", out_path.resolve())
